# Módulo 4 · Clase 4 — Diseño de bases de datos

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema | Qué aprenderás |
|---|------|----------------|
| 1 | Proceso de diseño | Conceptual → Lógico → Físico |
| 2 | Tipos de relaciones | 1:1, 1:N, N:M |
| 3 | Primera Forma Normal (1FN) | Sin grupos repetitivos |
| 4 | Segunda Forma Normal (2FN) | Sin dependencias parciales |
| 5 | Tercera Forma Normal (3FN) | Sin dependencias transitivas |
| 6 | Cuándo desnormalizar | Trade-offs en la práctica |
| 7 | Práctica | Diseñar la BD de un sistema universitario |

---
## 1. El proceso de diseño

```
┌─────────────────────────────────────────────────────────────┐
│ 1. ANÁLISIS DE REQUISITOS                                   │
│    «¿Qué información necesitamos almacenar y para qué?»     │
└───────────────────────────┬─────────────────────────────────┘
                            ▼
┌─────────────────────────────────────────────────────────────┐
│ 2. MODELO CONCEPTUAL — Diagrama ER                          │
│    Entidades, atributos y relaciones (sin SQL todavía)      │
└───────────────────────────┬─────────────────────────────────┘
                            ▼
┌─────────────────────────────────────────────────────────────┐
│ 3. MODELO LÓGICO — Tablas normalizadas                      │
│    Definir tablas, columnas, tipos, claves, restricciones   │
└───────────────────────────┬─────────────────────────────────┘
                            ▼
┌─────────────────────────────────────────────────────────────┐
│ 4. MODELO FÍSICO — DDL específico del motor                 │
│    CREATE TABLE, índices, particiones, etc.                 │
└─────────────────────────────────────────────────────────────┘
```

En esta clase nos centramos en los modelos **conceptual** y **lógico**: cómo pensar el diseño antes de escribir SQL.

---
## 2. Tipos de relaciones

### Relación 1:1 (uno a uno)

Cada fila de A corresponde a como máximo una fila de B, y viceversa.

```
empleados ──── 1:1 ──── fichas_medicas

Implementación: FK en la tabla con más registros opcionales, con UNIQUE

empleados(id, nombre, ...)
fichas_medicas(id, empleado_id UNIQUE, grupo_sanguineo, ...)
```

### Relación 1:N (uno a muchos) — la más común

Una fila de A puede tener muchas filas de B relacionadas, pero cada fila de B pertenece a solo una de A.

```
clientes ──── 1:N ──── pedidos

Un cliente → muchos pedidos
Un pedido  → un cliente

Implementación: FK en la tabla del lado «N»
pedidos(id, cliente_id, ...)   ← FK en pedidos
```

### Relación N:M (muchos a muchos)

Una fila de A puede relacionarse con muchas de B, y una de B con muchas de A.

```
estudiantes ──── N:M ──── asignaturas

Un estudiante → muchas asignaturas
Una asignatura → muchos estudiantes

Implementación: TABLA INTERMEDIA (tabla puente)

estudiantes(id, nombre)
asignaturas(id, nombre)
matriculas(estudiante_id, asignatura_id, fecha, nota)  ← tabla puente
         PK compuesta: (estudiante_id, asignatura_id)
```

---
## 3. Normalización

La normalización es el proceso de organizar las tablas para reducir la **redundancia** y mejorar la **integridad** de los datos.

Trabajaremos con esta tabla desnormalizada como ejemplo:

```
TABLA: pedidos_raw (tabla plana, todo en una)
┌────────┬───────────┬─────────────────┬─────────────┬────────────────────┬─────────┬──────────┬──────┐
│ped_id  │cli_nombre │cli_email        │ cli_ciudad  │ productos           │cantidad │precio    │fecha │
├────────┼───────────┼─────────────────┼─────────────┼────────────────────┼─────────┼──────────┼──────┤
│ 1      │ Ana       │ ana@mail.com    │ Madrid      │ Teclado, Ratón      │ 1,1     │ 85,45    │01/01 │
│ 2      │ Ana       │ ana@mail.com    │ Madrid      │ Smartphone          │ 1       │ 599      │15/01 │
│ 3      │ Carlos    │ carlos@mail.com │ Barcelona   │ Monitor, Teclado    │ 1,2     │ 450,85   │20/01 │
└────────┴───────────┴─────────────────┴─────────────┴────────────────────┴─────────┴──────────┴──────┘

Problemas:
❌ «Ana» y su email aparecen en varias filas (redundancia)
❌ «productos» y «cantidad» contienen listas separadas por comas (grupos repetitivos)
❌ Si Ana cambia de email, hay que actualizar varias filas
❌ Imposible consultar «pedidos que incluyen Teclado» con SQL
```

---
## 3. Primera Forma Normal (1FN)

**Reglas de la 1FN:**
1. Cada celda contiene un **único valor atómico** (no listas, no conjuntos).
2. Cada columna contiene valores del **mismo tipo**.
3. Cada fila es **única** (hay una clave primaria).

```
ANTES de 1FN (violación: varios productos en una celda):
┌───────┬──────────┬─────────────────────┬─────────┐
│ped_id │ cliente  │ productos            │cantidad │
├───────┼──────────┼─────────────────────┼─────────┤
│  1    │ Ana      │ Teclado, Ratón       │ 1, 1    │

DESPUÉS de 1FN (una fila por producto):
┌───────┬──────────┬───────────┬─────────┐
│ped_id │ cliente  │ producto  │cantidad │
├───────┼──────────┼───────────┼─────────┤
│  1    │ Ana      │ Teclado   │  1      │
│  1    │ Ana      │ Ratón     │  1      │
└───────┴──────────┴───────────┴─────────┘

PK: (ped_id, producto)  — clave primaria compuesta
```

---
## 4. Segunda Forma Normal (2FN)

Requisito previo: estar en 1FN.

**Regla de la 2FN:** Todos los atributos no clave deben depender de **toda** la clave primaria, no de una parte de ella.

Solo aplica cuando la PK es compuesta.

```
Tabla en 1FN con PK = (ped_id, producto_id):
┌────────┬────────────┬──────────┬───────────┬──────────────┬──────────┐
│ped_id  │producto_id │ cantidad │ cli_nombre│ prod_nombre  │ precio   │
├────────┼────────────┼──────────┼───────────┼──────────────┼──────────┤
│  1     │    5       │   1      │ Ana       │ Teclado      │  85.00   │

Dependencias:
  cantidad    → depende de (ped_id, producto_id) ✅ — depende de TODA la PK
  cli_nombre  → depende solo de ped_id ❌ — dependencia PARCIAL
  prod_nombre → depende solo de producto_id ❌ — dependencia PARCIAL
  precio      → depende solo de producto_id ❌ — dependencia PARCIAL

SOLUCIÓN: separar en tres tablas
  pedidos(ped_id, cliente_id, fecha)
  productos(producto_id, nombre, precio)
  lineas_pedido(ped_id, producto_id, cantidad)   ← solo lo que depende de TODA la PK
```

---
## 5. Tercera Forma Normal (3FN)

Requisito previo: estar en 2FN.

**Regla de la 3FN:** No debe haber dependencias transitivas. Un atributo no clave no debe depender de otro atributo no clave.

```
Tabla en 2FN:
clientes(id, nombre, ciudad, codigo_postal, pais)

Dependencias:
  nombre, ciudad → dependen de id ✅
  pais           → depende de codigo_postal ❌  (dependencia TRANSITIVA)
                   id → codigo_postal → pais

SOLUCIÓN:
  clientes(id, nombre, ciudad, codigo_postal)
  codigos_postales(codigo_postal, pais, provincia)  ← extraer la dependencia transitiva
```

**Regla práctica:** Si el dato X determina el dato Y, y ninguno de los dos es la PK, extrae una nueva tabla.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

conn = sqlite3.connect(":memory:")
cur  = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON")

# ── TABLA DESNORMALIZADA (sin ninguna forma normal) ──────────────
cur.execute("""
    CREATE TABLE pedidos_plano (
        pedido_id     INTEGER,
        cli_nombre    TEXT,
        cli_email     TEXT,
        cli_ciudad    TEXT,
        prod_nombres  TEXT,     -- ❌ lista: "Teclado, Ratón"
        cantidades    TEXT,     -- ❌ lista: "1, 1"
        precios       TEXT,     -- ❌ lista: "85, 45"
        fecha         TEXT
    )
""")

cur.executemany("INSERT INTO pedidos_plano VALUES (?,?,?,?,?,?,?,?)", [
    (1, 'Ana', 'ana@mail.com', 'Madrid',    'Teclado, Ratón',   '1, 1', '85, 45', '2025-01-10'),
    (2, 'Ana', 'ana@mail.com', 'Madrid',    'Smartphone',       '1',    '599',    '2025-01-15'),
    (3, 'Carlos', 'c@mail.com','Barcelona', 'Monitor, Teclado', '1, 2', '450, 85','2025-01-20'),
])

print("Tabla desnormalizada:")
display(pd.read_sql_query("SELECT * FROM pedidos_plano", conn))
print("\nProblemas: listas en celdas, datos de cliente repetidos.")

In [ ]:
# ── ESQUEMA NORMALIZADO (3FN) ────────────────────────────────────
cur.executescript("""
    CREATE TABLE clientes (
        id     INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        email  TEXT NOT NULL UNIQUE,
        ciudad TEXT
    );
    CREATE TABLE productos (
        id     INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        precio REAL NOT NULL
    );
    CREATE TABLE pedidos (
        id         INTEGER PRIMARY KEY,
        cliente_id INTEGER NOT NULL REFERENCES clientes(id),
        fecha      TEXT
    );
    CREATE TABLE lineas_pedido (
        pedido_id   INTEGER NOT NULL REFERENCES pedidos(id),
        producto_id INTEGER NOT NULL REFERENCES productos(id),
        cantidad    INTEGER NOT NULL,
        precio_ud   REAL    NOT NULL,
        PRIMARY KEY (pedido_id, producto_id)
    );
""")

# Poblar
cur.executemany("INSERT INTO clientes VALUES (?,?,?,?)", [
    (1, 'Ana',    'ana@mail.com', 'Madrid'),
    (2, 'Carlos', 'c@mail.com',   'Barcelona'),
])
cur.executemany("INSERT INTO productos VALUES (?,?,?)", [
    (1, 'Teclado',    85.00),
    (2, 'Ratón',      45.00),
    (3, 'Smartphone', 599.00),
    (4, 'Monitor',    450.00),
])
cur.executemany("INSERT INTO pedidos VALUES (?,?,?)", [
    (1, 1, '2025-01-10'),
    (2, 1, '2025-01-15'),
    (3, 2, '2025-01-20'),
])
cur.executemany("INSERT INTO lineas_pedido VALUES (?,?,?,?)", [
    (1, 1, 1, 85.00),
    (1, 2, 1, 45.00),
    (2, 3, 1, 599.00),
    (3, 4, 1, 450.00),
    (3, 1, 2, 85.00),
])
conn.commit()

# Ahora la misma información es perfectamente consultable
print("Esquema normalizado — detalle de pedidos:")
display(pd.read_sql_query("""
    SELECT p.id AS pedido, c.nombre AS cliente, prod.nombre AS producto,
           lp.cantidad, lp.precio_ud,
           ROUND(lp.cantidad * lp.precio_ud, 2) AS subtotal
    FROM lineas_pedido lp
    JOIN pedidos   p    ON lp.pedido_id   = p.id
    JOIN clientes  c    ON p.cliente_id   = c.id
    JOIN productos prod ON lp.producto_id = prod.id
    ORDER BY p.id, prod.nombre
""", conn))

conn.close()

---
## 6. Cuándo desnormalizar

La normalización es la regla general, pero tiene excepciones:

| Situación | Puede justificar desnormalizar |
|-----------|-------------------------------|
| Consultas de lectura muy frecuentes | Reducir JOINs costosos |
| Data warehouses / analítica | Las tablas planas son más rápidas para agregaciones |
| Columnas que raramente cambian | La redundancia tiene bajo coste de mantenimiento |
| Auditoría / histórico | Guardar el precio en el momento de la venta |

```
Ejemplo de desnormalización justificada:

lineas_pedido.precio_ud  ← guardamos el precio en el momento de la venta
                           (aunque ya esté en productos.precio)

¿Por qué? Si mañana cambia el precio del producto, el importe del
pedido histórico debe reflejar el precio que pagó el cliente, no el actual.
```

> Regla práctica: **normaliza primero, desnormaliza solo cuando el rendimiento lo exige** y puedes medir el beneficio.

---
## Diagrama ER — ejemplo completo

```
┌─────────────┐        ┌─────────────────┐        ┌─────────────┐
│  clientes   │        │    pedidos      │        │  productos  │
├─────────────┤        ├─────────────────┤        ├─────────────┤
│ PK id       │──1──┐  │ PK id           │  ┌──N──│ PK id       │
│    nombre   │     └─N│ FK cliente_id   │  │     │    nombre   │
│    email    │        │    fecha        │  │     │    precio   │
│    ciudad   │        │    estado       │  │     │    stock    │
└─────────────┘        └────────┬────────┘  │     │ FK cat_id   │
                                │            │     └──────┬──────┘
                               1│            │N           │N
                                ▼            │            │
                       ┌─────────────────┐  │     ┌──────┴──────┐
                       │ lineas_pedido   │◄─┘     │ categorias  │
                       ├─────────────────┤        ├─────────────┤
                       │ PK FK pedido_id │        │ PK id       │
                       │ PK FK prod_id  │        │    nombre   │
                       │    cantidad    │        └─────────────┘
                       │    precio_ud   │
                       └─────────────────┘

Relaciones:
clientes ──1:N──► pedidos
pedidos ──N:M──► productos  (a través de lineas_pedido)
productos ──N:1──► categorias
```

---
## 7. Práctica — Sistema universitario

---

### Ejercicio 1 — Identificar las formas normales

**Enunciado:** Analiza esta tabla y determina qué formas normales viola:

```
tabla: inscripciones
┌──────────┬──────────────┬────────────────────────────┬───────┬───────────────┬──────────────────┐
│ alumno_id│ alumno_nombre│ asignaturas                 │notas  │prof_nombre    │ departamento     │
├──────────┼──────────────┼────────────────────────────┼───────┼───────────────┼──────────────────┤
│  1       │ María        │ Matemáticas, Física         │ 8, 7  │ García, López │ Ciencias, Ciencias│
│  2       │ Juan         │ Informática                 │ 9     │ Martínez      │ Tecnología       │
│  3       │ María        │ Informática                 │ 6     │ Martínez      │ Tecnología       │
└──────────┴──────────────┴────────────────────────────┴───────┴───────────────┴──────────────────┘
```

Responde en una celda Markdown:
1. ¿Qué viola la 1FN?
2. ¿Qué viola la 2FN (asume PK compuesta)?
3. ¿Qué viola la 3FN?
4. Propón el esquema normalizado.

✏️ **TU RESPUESTA AQUÍ**

1FN:

2FN:

3FN:

Esquema propuesto:


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

**1FN — Viola** porque:
- `asignaturas`, `notas`, `prof_nombre`, `departamento` contienen listas de valores en una sola celda (no son atómicos).

**2FN — Viola** (asumiendo PK = (alumno_id, asignatura_id)) porque:
- `alumno_nombre` depende solo de `alumno_id` → dependencia parcial.
- `prof_nombre` y `departamento` dependen solo de `asignatura_id` → dependencia parcial.

**3FN — Viola** porque:
- `departamento` depende de `prof_nombre`, no directamente de la PK → dependencia transitiva.

**Esquema normalizado (3FN):**
```
departamentos(id, nombre)
profesores(id, nombre, departamento_id FK)
alumnos(id, nombre, email)
asignaturas(id, nombre, profesor_id FK)
inscripciones(alumno_id FK, asignatura_id FK, nota)  PK(alumno_id, asignatura_id)
```
</details>

---
### Ejercicio 2 — Implementar el esquema universitario

**Enunciado:** Implementa en SQLite el esquema normalizado del ejercicio anterior. Luego inserta datos de prueba:
- Tres departamentos, cuatro profesores, seis alumnos, cinco asignaturas.
- Al menos diez inscripciones con notas variadas.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import sqlite3, pandas as pd
from pathlib import Path


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import sqlite3, pandas as pd
from pathlib import Path

conn = sqlite3.connect(Path.home() / "universidad.db")
cur  = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON")

cur.executescript("""
    DROP TABLE IF EXISTS inscripciones;
    DROP TABLE IF EXISTS asignaturas;
    DROP TABLE IF EXISTS alumnos;
    DROP TABLE IF EXISTS profesores;
    DROP TABLE IF EXISTS departamentos;

    CREATE TABLE departamentos (
        id     INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL UNIQUE
    );
    CREATE TABLE profesores (
        id              INTEGER PRIMARY KEY,
        nombre          TEXT NOT NULL,
        departamento_id INTEGER REFERENCES departamentos(id)
    );
    CREATE TABLE alumnos (
        id     INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        email  TEXT NOT NULL UNIQUE
    );
    CREATE TABLE asignaturas (
        id          INTEGER PRIMARY KEY,
        nombre      TEXT NOT NULL,
        creditos    INTEGER DEFAULT 6,
        profesor_id INTEGER REFERENCES profesores(id)
    );
    CREATE TABLE inscripciones (
        alumno_id     INTEGER NOT NULL REFERENCES alumnos(id),
        asignatura_id INTEGER NOT NULL REFERENCES asignaturas(id),
        nota          REAL CHECK(nota BETWEEN 0 AND 10),
        PRIMARY KEY (alumno_id, asignatura_id)
    );
""")

cur.executemany("INSERT INTO departamentos VALUES (?,?)", [
    (1,'Ciencias'),(2,'Tecnología'),(3,'Humanidades')
])
cur.executemany("INSERT INTO profesores VALUES (?,?,?)", [
    (1,'García',1),(2,'López',1),(3,'Martínez',2),(4,'Ruiz',3)
])
cur.executemany("INSERT INTO alumnos VALUES (?,?,?)", [
    (1,'María','maria@uni.es'),(2,'Juan','juan@uni.es'),
    (3,'Laura','laura@uni.es'),(4,'Pedro','pedro@uni.es'),
    (5,'Sofía','sofia@uni.es'),(6,'Marcos','marcos@uni.es'),
])
cur.executemany("INSERT INTO asignaturas VALUES (?,?,?,?)", [
    (1,'Matemáticas',6,1),(2,'Física',6,2),(3,'Informática',6,3),
    (4,'Literatura',6,4),(5,'Estadística',4,1),
])
cur.executemany("INSERT INTO inscripciones VALUES (?,?,?)", [
    (1,1,8.5),(1,2,7.0),(1,3,6.5),
    (2,3,9.0),(2,5,7.5),
    (3,1,5.0),(3,4,8.0),
    (4,2,4.5),(4,3,7.0),(4,5,6.0),
    (5,1,9.5),(5,3,8.0),
    (6,4,7.5),(6,2,6.0),
])
conn.commit()
print("Base de datos universitaria creada")
```
</details>

---
### Ejercicio 3 — Consultas sobre el esquema

**Enunciado:** Con la BD universitaria del ejercicio anterior:
1. Lista los alumnos con nota media ≥ 7.
2. Cuenta cuántos alumnos están inscritos en cada asignatura.
3. Muestra qué asignaturas imparte cada departamento.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
def q(sql): return pd.read_sql_query(sql, conn)

# 1. Alumnos con media >= 7
display(q("""
    SELECT a.nombre, ROUND(AVG(i.nota), 2) AS nota_media
    FROM alumnos a JOIN inscripciones i ON a.id = i.alumno_id
    GROUP BY a.id, a.nombre
    HAVING AVG(i.nota) >= 7
    ORDER BY nota_media DESC
"""))

# 2. Alumnos por asignatura
display(q("""
    SELECT asig.nombre, COUNT(i.alumno_id) AS alumnos
    FROM asignaturas asig
    LEFT JOIN inscripciones i ON asig.id = i.asignatura_id
    GROUP BY asig.id, asig.nombre
    ORDER BY alumnos DESC
"""))

# 3. Asignaturas por departamento
display(q("""
    SELECT d.nombre AS departamento, GROUP_CONCAT(asig.nombre, ', ') AS asignaturas
    FROM departamentos d
    JOIN profesores p ON p.departamento_id = d.id
    JOIN asignaturas asig ON asig.profesor_id = p.id
    GROUP BY d.id, d.nombre
"""))
```
</details>

---
### Ejercicio 4 — Identificar relaciones

**Enunciado:** Para cada par de entidades, identifica el tipo de relación (1:1, 1:N, N:M) y explica por qué:

1. `paciente` — `historial_medico`
2. `autor` — `libro`
3. `vuelo` — `pasajero`
4. `empleado` — `departamento`
5. `producto` — `etiqueta` (tags)

✏️ **TU RESPUESTA AQUÍ**

1. paciente — historial_medico:
2. autor — libro:
3. vuelo — pasajero:
4. empleado — departamento:
5. producto — etiqueta:


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

1. **paciente — historial_medico: 1:1** — cada paciente tiene un único historial y cada historial pertenece a un único paciente.

2. **autor — libro: 1:N** — un autor puede escribir muchos libros, pero cada libro tiene un único autor principal. (Si asumimos coautoría, sería N:M.)

3. **vuelo — pasajero: N:M** — un vuelo lleva a muchos pasajeros y un pasajero puede tomar muchos vuelos. Tabla puente: `reservas(vuelo_id, pasajero_id, asiento, ...)`.

4. **empleado — departamento: N:1** — muchos empleados pertenecen a un departamento, pero cada empleado pertenece a uno solo. FK en `empleados.departamento_id`.

5. **producto — etiqueta: N:M** — un producto puede tener varias etiquetas y la misma etiqueta puede aplicarse a varios productos. Tabla puente: `producto_etiquetas(producto_id, etiqueta_id)`.
</details>

---
### Ejercicio 5 (reto) — Diseñar desde cero

**Enunciado:** Diseña e implementa la base de datos para una **plataforma de streaming de música** con los siguientes requisitos:

- Los **artistas** pueden colaborar en canciones (un artista puede tener muchas canciones; una canción puede tener varios artistas).
- Las **canciones** pertenecen a **álbumes**. Un álbum tiene un artista principal, pero puede incluir colaboraciones.
- Los **usuarios** pueden crear **listas de reproducción** y añadir canciones.
- El sistema registra las **reproducciones** (qué usuario escuchó qué canción y cuándo).

Implementa el esquema en SQLite, inserta datos de prueba y escribe tres consultas útiles (p. ej., artistas más escuchados, canciones más populares, usuario con más reproducciones).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import sqlite3, pandas as pd
from pathlib import Path


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import sqlite3, pandas as pd
from pathlib import Path

conn = sqlite3.connect(Path.home() / "streaming.db")
cur  = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON")

cur.executescript("""
    DROP TABLE IF EXISTS reproducciones;
    DROP TABLE IF EXISTS playlist_canciones;
    DROP TABLE IF EXISTS playlists;
    DROP TABLE IF EXISTS cancion_artistas;
    DROP TABLE IF EXISTS canciones;
    DROP TABLE IF EXISTS albumes;
    DROP TABLE IF EXISTS usuarios;
    DROP TABLE IF EXISTS artistas;

    CREATE TABLE artistas (
        id     INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        pais   TEXT
    );
    CREATE TABLE albumes (
        id           INTEGER PRIMARY KEY,
        titulo       TEXT NOT NULL,
        anio         INTEGER,
        artista_id   INTEGER REFERENCES artistas(id)
    );
    CREATE TABLE canciones (
        id         INTEGER PRIMARY KEY,
        titulo     TEXT NOT NULL,
        duracion_s INTEGER,
        album_id   INTEGER REFERENCES albumes(id)
    );
    CREATE TABLE cancion_artistas (
        cancion_id INTEGER REFERENCES canciones(id),
        artista_id INTEGER REFERENCES artistas(id),
        rol        TEXT DEFAULT 'principal',
        PRIMARY KEY (cancion_id, artista_id)
    );
    CREATE TABLE usuarios (
        id     INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        email  TEXT UNIQUE
    );
    CREATE TABLE playlists (
        id         INTEGER PRIMARY KEY,
        nombre     TEXT NOT NULL,
        usuario_id INTEGER REFERENCES usuarios(id)
    );
    CREATE TABLE playlist_canciones (
        playlist_id INTEGER REFERENCES playlists(id),
        cancion_id  INTEGER REFERENCES canciones(id),
        posicion    INTEGER,
        PRIMARY KEY (playlist_id, cancion_id)
    );
    CREATE TABLE reproducciones (
        id          INTEGER PRIMARY KEY,
        usuario_id  INTEGER REFERENCES usuarios(id),
        cancion_id  INTEGER REFERENCES canciones(id),
        fecha       TEXT DEFAULT (datetime('now'))
    );
""")

# Datos
cur.executemany("INSERT INTO artistas VALUES (?,?,?)", [
    (1,'Taylor Swift','US'),(2,'Bad Bunny','PR'),
    (3,'Rosalía','ES'),(4,'The Weeknd','CA'),
])
cur.executemany("INSERT INTO albumes VALUES (?,?,?,?)", [
    (1,'Midnights',2022,1),(2,'Un Verano Sin Ti',2022,2),
    (3,'El Mal Querer',2018,3),(4,'Dawn FM',2022,4),
])
cur.executemany("INSERT INTO canciones VALUES (?,?,?,?)", [
    (1,'Anti-Hero',197,1),(2,'Lavender Haze',202,1),(3,'Midnight Rain',174,1),
    (4,'Tití Me Preguntó',238,2),(5,'Ojitos Lindos',224,2),
    (6,'Malamente',215,3),(7,'Bagdad',259,3),
    (8,'Sacrifice',189,4),(9,'Less Than Zero',175,4),
])
cur.executemany("INSERT INTO cancion_artistas VALUES (?,?,?)", [
    (1,1,'principal'),(2,1,'principal'),(3,1,'principal'),
    (4,2,'principal'),(5,2,'principal'),(5,3,'colaboración'),  # Ojitos Lindos: Bad Bunny + Rosalía
    (6,3,'principal'),(7,3,'principal'),
    (8,4,'principal'),(9,4,'principal'),
])
cur.executemany("INSERT INTO usuarios VALUES (?,?,?)", [
    (1,'Ana','ana@demo.com'),(2,'Carlos','carlos@demo.com'),(3,'Lucía','lucia@demo.com'),
])
cur.executemany("INSERT INTO playlists VALUES (?,?,?)", [
    (1,'Favoritas',1),(2,'Workout',2),(3,'Relax',3),
])
cur.executemany("INSERT INTO playlist_canciones VALUES (?,?,?)", [
    (1,1,1),(1,6,2),(1,8,3),
    (2,4,1),(2,5,2),(2,8,3),
    (3,2,1),(3,7,2),(3,9,3),
])
# Simular reproducciones
for uid, cid in [(1,1),(1,1),(1,6),(2,4),(2,4),(2,4),(2,5),(3,8),(3,9),(3,1),(1,8),(2,8)]:
    cur.execute("INSERT INTO reproducciones (usuario_id, cancion_id) VALUES (?,?)", (uid,cid))
conn.commit()

def q(sql): return pd.read_sql_query(sql, conn)

print("=== Top canciones ===")
display(q("""
    SELECT c.titulo, COUNT(*) AS reproducciones
    FROM reproducciones r JOIN canciones c ON r.cancion_id = c.id
    GROUP BY c.id, c.titulo ORDER BY reproducciones DESC LIMIT 5
"""))

print("\n=== Artistas más escuchados ===")
display(q("""
    SELECT a.nombre, COUNT(*) AS reproducciones
    FROM reproducciones r
    JOIN cancion_artistas ca ON r.cancion_id = ca.cancion_id
    JOIN artistas a ON ca.artista_id = a.id
    GROUP BY a.id, a.nombre ORDER BY reproducciones DESC
"""))

print("\n=== Usuario más activo ===")
display(q("""
    SELECT u.nombre, COUNT(*) AS reproducciones
    FROM reproducciones r JOIN usuarios u ON r.usuario_id = u.id
    GROUP BY u.id, u.nombre ORDER BY reproducciones DESC LIMIT 1
"""))
```
</details>

---
## Resumen de la clase

```
TIPOS DE RELACIONES
  1:1  → FK + UNIQUE en la tabla secundaria
  1:N  → FK en la tabla del lado «N»
  N:M  → Tabla puente con PK compuesta (fk1, fk2)

FORMAS NORMALES
  1FN → Valores atómicos + PK
  2FN → Sin dependencias parciales (solo con PK compuesta)
  3FN → Sin dependencias transitivas entre no-clave

PROCESO DE DISEÑO
  1. Identificar entidades (sustantivos del dominio)
  2. Identificar relaciones (verbos: «tiene», «pertenece a», «compra»)
  3. Identificar atributos de cada entidad
  4. Normalizar hasta 3FN
  5. Desnormalizar solo si hay evidencia de problema de rendimiento
```

## ¿Qué viene en la próxima clase?

En la **Clase 5** conectamos Python con bases de datos SQL: el módulo `sqlite3` de la biblioteca estándar y `SQLAlchemy` como ORM.